# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

print(df.shape)
df.head()

(10910, 26)


,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


In [2]:
#lowercase and underscores
df.columns = df.columns.str.lower().str.replace(" ", "_")

# Remove the useless index column
df = df.drop(columns=["unnamed:_0"])

# Turn the date text into a real date, then take the month
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], format="%m/%d/%y")
df["month"] = df["effective_to_date"].dt.month

print(df.columns)
print(df["month"].unique())

Index(['customer', 'state', 'customer_lifetime_value', 'response', 'coverage',
       'education', 'effective_to_date', 'employmentstatus', 'gender',
       'income', 'location_code', 'marital_status', 'monthly_premium_auto',
       'months_since_last_claim', 'months_since_policy_inception',
       'number_of_open_complaints', 'number_of_policies', 'policy_type',
       'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount',
       'vehicle_class', 'vehicle_size', 'vehicle_type', 'month'],
      dtype='str')
[2 1]


In [3]:
low_claim_yes = df[(df["total_claim_amount"] < 1000) & (df["response"] == "Yes")]

print(low_claim_yes.shape)
low_claim_yes.head()

(1399, 26)


,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,49078,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
8,FM55990,California,5989.773931,Yes,Premium,College,2011-01-19,Employed,M,66839,...,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN,1
15,CW49887,California,4626.801093,Yes,Basic,Master,2011-01-16,Employed,F,79487,...,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN,1
19,NJ54277,California,3746.751625,Yes,Extended,College,2011-02-26,Employed,F,41479,...,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A,2
27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2011-02-28,Employed,F,63774,...,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN,2


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [13]:
# Customers who responded Yes
yes_customers = df[df["response"] == "Yes"]

# Value: average premium and CLV for Yes customers
premium_yes = yes_customers.pivot_table(index="policy_type", columns="gender",
                                        values="monthly_premium_auto", aggfunc="mean").round(2)

clv_yes = yes_customers.pivot_table(index="policy_type", columns="gender",
                                    values="customer_lifetime_value", aggfunc="mean").round(2)

# Risk: average claim for Yes customers vs all customers
claims_yes = yes_customers.pivot_table(index="policy_type", columns="gender",
                                       values="total_claim_amount", aggfunc="mean").round(2)

claims_all = df.pivot_table(index="policy_type", columns="gender",
                            values="total_claim_amount", aggfunc="mean").round(2)

print("Average monthly premium (Yes):\n", premium_yes)
print("\nAverage CLV (Yes):\n", clv_yes)
print("\nAverage claim (Yes):\n", claims_yes)
print("\nAverage claim (all customers):\n", claims_all)


summary = yes_customers.groupby(["policy_type", "gender"])[["monthly_premium_auto",
                                                           "customer_lifetime_value",
                                                           "total_claim_amount"]].mean().round(2)
summary = summary.sort_values("customer_lifetime_value", ascending=False)
summary

Average monthly premium (Yes):
 gender              F      M
policy_type                 
Corporate Auto  94.30  92.19
Personal Auto   99.00  91.09
Special Auto    92.31  86.34

Average CLV (Yes):
 gender                F        M
policy_type                     
Corporate Auto  7712.63  7944.47
Personal Auto   8339.79  7448.38
Special Auto    7691.58  8247.09

Average claim (Yes):
 gender               F       M
policy_type                   
Corporate Auto  433.74  408.58
Personal Auto   452.97  457.01
Special Auto    453.28  429.53

Average claim (all customers):
 gender               F       M
policy_type                   
Corporate Auto  397.80  462.22
Personal Auto   413.24  459.92
Special Auto    458.14  420.36


monthly_premium_auto  customer_lifetime_value  \
policy_type    gender                                                  
Personal Auto  F                      99.00                  8339.79   
Special Auto   M                      86.34                  8247.09   
Corporate Auto M                      92.19                  7944.47   
               F                      94.30                  7712.63   
Special Auto   F                      92.31                  7691.58   
Personal Auto  M                      91.09                  7448.38   

                       total_claim_amount  
policy_type    gender                      
Personal Auto  F                   452.97  
Special Auto   M                   429.53  
Corporate Auto M                   408.58  
               F                   433.74  
Special Auto   F                   453.28  
Personal Auto  M                   457.01

Conclusions

Note: the dataset’s column is monthly_premium_auto. There’s no monthly_premium column, so this is the one used.

Most profitable: Personal Auto women have the highest CLV (8,339.79) and the highest premium (99.00), with average claims (452.97).
Lowest risk: Corporate Auto men have the lowest average claims (408.58) and a good CLV (7,944.47).
Also strong: Special Auto men have a high CLV (8,247.09) with below-average claims (429.53), even though they pay the lowest premium (86.34).
Least attractive: Personal Auto men have the lowest CLV (7,448.38) and the highest claims (457.01).
Differences are small overall (claims range from about 409 to 457), so no segment is dramatically riskier than the others.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [5]:
customers_per_state = df["state"].value_counts()
print(customers_per_state)

big_states = customers_per_state[customers_per_state > 500]
print(big_states)

state
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: count, dtype: int64
state
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: count, dtype: int64


4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [11]:
#Group by state and month and count policies
by_state_month = df.groupby(["state", "month"])["customer"].count().reset_index(name="policies_sold")

#Top 3 states from the grouped totals
total_by_state = by_state_month.groupby("state")["policies_sold"].sum().sort_values(ascending=False)
top3_states = total_by_state.head(3).index
print(top3_states)

#Keep the top 3 and reshape: states as rows, months as columns
top3_by_month = by_state_month[by_state_month["state"].isin(top3_states)]

top3_table = top3_by_month.pivot_table(index="state", columns="month",
                                       values="policies_sold").astype(int)
top3_table = top3_table.loc[top3_states]   # keep the order from highest to lowest
print(top3_table)

Index(['California', 'Oregon', 'Arizona'], dtype='str', name='state')
month          1     2
state                 
California  1918  1634
Oregon      1565  1344
Arizona     1008   929


In [14]:
clv_summary = df.groupby(["education", "gender"])["customer_lifetime_value"].agg(["max", "min", "median"]).round(2)
clv_summary

max      min   median
education            gender                            
Bachelor             F       73225.96  1904.00  5640.51
                     M       67907.27  1898.01  5548.03
College              F       61850.19  1898.68  5623.61
                     M       61134.68  1918.12  6005.85
Doctor               F       44856.11  2395.57  5332.46
                     M       32677.34  2267.60  5577.67
High School or Below F       55277.45  2144.92  6039.55
                     M       83325.38  1940.98  6286.73
Master               F       51016.07  2417.78  5729.86
                     M       50568.26  2272.31  5579.10

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [7]:
policies_state_month = df.pivot_table(index="state",
                                      columns="month",
                                      values="customer",
                                      aggfunc="count")
print(policies_state_month)

month          1     2
state                 
Arizona     1008   929
California  1918  1634
Nevada       551   442
Oregon      1565  1344
Washington   463   425


6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [8]:
# # Group by state and month, count, sort descending
# by_state_month = df.groupby(["state", "month"])["customer"].count().reset_index(name="policies_sold")
# by_state_month = by_state_month.sort_values("policies_sold", ascending=False)

# #Top 3 states overall
# top3_states = df["state"].value_counts().head(3).index
# print(top3_states)   # California, Oregon, Arizona

# # Keep only those states
# top3_by_month = by_state_month[by_state_month["state"].isin(top3_states)]
# print(top3_by_month)

Index(['California', 'Oregon', 'Arizona'], dtype='str', name='state')
        state  month  policies_sold
2  California      1           1918
3  California      2           1634
6      Oregon      1           1565
7      Oregon      2           1344
0     Arizona      1           1008
1     Arizona      2            929


In [16]:
by_state_month = df.groupby(["state", "month"])["customer"].count().reset_index(name="policies_sold")
by_state_month = by_state_month.sort_values("policies_sold", ascending=False)

total_by_state = by_state_month.groupby("state")["policies_sold"].sum().sort_values(ascending=False)
top3_states = total_by_state.head(3).index

top3_by_month = by_state_month[by_state_month["state"].isin(top3_states)]

top3_table = top3_by_month.pivot_table(index="state", columns="month",
                                       values="policies_sold", fill_value=0).astype(int)
top3_table = top3_table.loc[top3_states]
top3_table


#California, Oregon and Arizona are the top 3 states. All three sold more policies in January than in February.

month,1,2
state,,
California,1918,1634
Oregon,1565,1344
Arizona,1008,929


7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [9]:
# True if the customer said Yes, False otherwise
df["responded"] = df["response"] == "Yes"

response_rate = (df.groupby("sales_channel")["responded"].mean() * 100).round(2)
print(response_rate)

sales_channel
Agent          18.01
Branch         10.79
Call Center    10.32
Web            10.89
Name: responded, dtype: float64
